## Example of DEM with coherent errors with $d=5$ repetition code

In [1]:
import cirq
import numpy as np
import math
from IPython.display import display, Markdown


N_DATA_QUBITS = 5
N_AUX_QUBITS = 4
N_QUBITS = N_DATA_QUBITS + N_AUX_QUBITS
ANGLE = 0.01 * np.pi
circuit = cirq.Circuit()
for i in range(N_QUBITS):
    circuit.append(cirq.H(cirq.LineQubit(i)))

for i in range(0, N_QUBITS, 2):
    circuit.append(cirq.rz(ANGLE * 2.0)(cirq.LineQubit(i)))

for i in range(1, N_QUBITS, 2):
    circuit.append(cirq.CNOT(cirq.LineQubit(i), cirq.LineQubit(i - 1)))
    circuit.append(cirq.CNOT(cirq.LineQubit(i), cirq.LineQubit(i + 1)))

for i in range(1, N_QUBITS, 2):
    circuit.append(cirq.H(cirq.LineQubit(i)))

qubits = sorted(list(circuit.all_qubits()), reverse=True)

probs = np.absolute(circuit.final_state_vector(qubit_order=qubits)) ** 2

count_arr = np.zeros((N_AUX_QUBITS, N_AUX_QUBITS), dtype=float)
count_xor = np.zeros((N_AUX_QUBITS, N_AUX_QUBITS), dtype=float)

for p in range(len(probs)):
    bitstring = format(p, f"0{N_QUBITS}b")
    for i in range(0, N_AUX_QUBITS):
        for j in range(0, N_AUX_QUBITS):
            if (
                bitstring[N_QUBITS - 2 - 2 * i] == "1"
                and bitstring[N_QUBITS - 2 - 2 * j] == "1"
            ):
                count_arr[i, j] += probs[p]

for p in range(len(probs)):
    bitstring = format(p, f"0{N_QUBITS}b")
    for i in range(N_AUX_QUBITS):
        for j in range(N_AUX_QUBITS):
            if (
                bitstring[N_QUBITS - 2 - 2 * i] == "1"
                and bitstring[N_QUBITS - 2 - 2 * j] == "0"
                and i != j
            ) or (
                bitstring[N_QUBITS - 2 - 2 * i] == "0"
                and bitstring[N_QUBITS - 2 - 2 * j] == "1"
                and i != j
            ):
                count_xor[i, j] += probs[p]

p_arr = np.zeros((N_AUX_QUBITS, N_AUX_QUBITS), dtype=float)

for i in range(N_AUX_QUBITS):
    for j in range(N_AUX_QUBITS):
        if i != j:
            p_arr[i, j] = 0.5 - np.sqrt(
                0.25
                - (count_arr[i, j] - count_arr[i, i] * count_arr[j, j])
                / (
                    1.0
                    - 2.0 * (count_arr[i, i] + count_arr[j, j])
                    + 4.0 * count_arr[i, j]
                )
            )

for i in range(N_AUX_QUBITS):
    denom = 1.0
    for j in range(N_AUX_QUBITS):
        if i != j:
            denom *= 1.0 - 2.0 * p_arr[i, j]
    p_arr[i, i] = 0.5 + (count_arr[i, i] - 0.5) / denom


def calculate_angle(p):
    if p >= 0.0 and math.isnan(p) == False:
        return (1.0 / np.pi) * np.arcsin(np.sqrt(p))
    else:
        return math.nan


angle_arr = np.vectorize(calculate_angle)(p_arr)


print(f"Applied error is:{ANGLE / np.pi:.3f}π")
for i in range(N_AUX_QUBITS):
    for j in range(i, N_AUX_QUBITS):
        if angle_arr[i, j] != 0.0 and math.isnan(angle_arr[i, j]) == False:
            display(
                Markdown(
                    f"edge {i + 1} $\\leftrightarrow$ {j + 1}:\t $\\theta$=\t{angle_arr[i, j]:.3f}π\n"
                )
            )

Applied error is:0.010π


edge 1 $\leftrightarrow$ 1:	 $\theta$=	0.010π


edge 1 $\leftrightarrow$ 2:	 $\theta$=	0.010π


edge 2 $\leftrightarrow$ 3:	 $\theta$=	0.010π


edge 3 $\leftrightarrow$ 4:	 $\theta$=	0.010π


edge 4 $\leftrightarrow$ 4:	 $\theta$=	0.010π
